# E6 Colab — 27B 제로샷으로 train·dev 전체 진단 (A100 80GB, vLLM)

정답은 **여러 교육생이 일치해서 고른 답**입니다. 응답이 갈린 문항은 train/test에서 빠졌습니다.
그래서 모델이 사람과 다르게 고른 문항은 "라벨 오류"가 아니라, 대부분 **사람들이 해석하는 방식**
(일반인 상식, 눈에 띄는 쪽, 적힌 그대로)을 모델이 모르는 경우일 수 있습니다.

이 노트북은 QLoRA에 앞서 그 차이를 데이터로 봅니다.

| 단계 | 대상 | 알고 싶은 것 | 시간 |
| --- | --- | --- | ---: |
| 1 | train 6,714 | 새 기준선, 모델–사람 불일치 문항 | 약 22분 |
| 2 | dev 2,683 | dev가 진짜 사람 응답인지, 모델이 다수 응답을 얼마나 따르는지 | 약 9분 |
| 3 | train 6,714 + **합의 지시문** | "사람들이 가장 많이 고를 답"을 고르라고 하면 나아지는지 | 약 22분 |

모델 적재까지 합쳐 **약 1시간 10분**입니다. 단계마다 256문항씩 Drive에 저장하므로
세션이 끊기면 같은 셀을 다시 실행해 이어 갑니다. 시간이 모자라면 3단계는 다음 세션으로 미뤄도 됩니다.

## 준비물

- A100 80GB 런타임
- Drive의 `MyDrive/SSAFY/AI_Challenge2/ssafy-16-2-ai.zip`

공통 코드(`common_eval.py`)와 평가 분할은 이 노트북 안에 들어 있습니다.

## 평가 분할

E6부터는 학습을 하므로 평가 전용 분할을 둡니다. train 6,714 = **학습 5,414 + holdout 1,300**이고,
holdout에는 기존 val 672가 전부 들어 있습니다. 이 노트북은 학습이 없어 train 전체를 풀지만,
결과는 전체·holdout·val 672로 나눠 보여줍니다.

## 1. 설치

새 런타임에서 가장 먼저 실행하고, 끝나면 **런타임 → 세션 다시 시작** 후 2번 셀부터 실행하세요.
vLLM이 torch를 새 CUDA 빌드로 바꾸므로 새 프로세스에서 불러와야 합니다.

In [ ]:
!pip -q install -U vllm
!pip -q install -U "huggingface_hub[hf_xet]" pillow pandas
# vLLM이 바꾼 torch와 Colab 기본 torchaudio의 CUDA 버전이 어긋나 transformers import가 실패합니다.
!pip -q uninstall -y torchaudio

In [ ]:
import gc, json, math, os, re, shutil, sys, time
from math import comb
from pathlib import Path

import pandas as pd
import torch
import torchvision
import transformers
import vllm

print("vllm", vllm.__version__, "/ transformers", transformers.__version__)
print("torch", torch.__version__, f"(CUDA {torch.version.cuda})", "/ torchvision", torchvision.__version__)
try:
    import torchaudio  # noqa: F401
    raise RuntimeError("torchaudio가 남아 있습니다. `!pip uninstall -y torchaudio` 후 세션을 다시 시작하세요")
except ImportError:
    pass
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요"
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)} / {GPU_GIB:.1f} GiB")
assert GPU_GIB >= 70, "27B bf16(약 52 GiB)은 A100 80GB가 필요합니다"

## 2. Drive · 데이터 · 공통 코드

`common_eval.py`는 노트북 생성 시점의 저장소 원문을 그대로 씁니다(SHA-256 대조).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/SSAFY/AI_Challenge2")
ZIP_PATH = DRIVE_BASE / "ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"
assert ZIP_PATH.is_file(), f"zip을 찾지 못했습니다: {ZIP_PATH}"


def find_data_root():
    for parent, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d not in {"drive", ".cache", "sample_data", "models"}]
        if "test.csv" in files and "train.csv" in files:
            return Path(parent)
    return None


if find_data_root() is None:
    import subprocess
    subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", "-o/content/", str(ZIP_PATH)],
                   check=True, stdout=subprocess.DEVNULL)
DATA_ROOT = find_data_root()
assert DATA_ROOT is not None and (DATA_ROOT / "dev.csv").is_file(), "train/test/dev.csv를 찾지 못했습니다"

COMMON_EVAL_SRC = '"""실험 공통 코드.\n\n모든 실험(A: OCR+LLM, B: VLM 단독, C: OCR+VLM, D: LLM blind)이\n동일한 검증 분할 / 질문 프롬프트 / 출력 파서 / 분석 지표를 쓰도록 한곳에 모읍니다.\n각 노트북이 코드를 복사하면 조용히 어긋나므로 반드시 import 해서 사용하세요.\n"""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport pandas as pd\n\nDATA_DIR = Path("data")\nCHOICES = list("abcd")\nSYSTEM = ("You are a visual question answering assistant. "\n          "Answer with exactly one lowercase letter: a, b, c, or d. No explanation.")\n# 실험 A가 만든 기준 분할. 모든 실험은 이 val 672문항을 그대로 씁니다.\nBASELINE_RUN = Path("output/paddleocr_vl16_halftrain/full")\n\n\nANSWER_INSTRUCTION = "이미지를 보고 정답 선지의 영문 소문자 한 글자만 답하세요."\n# CoT에서는 "한 글자만" 지시와 "단계적으로 설명" 지시가 충돌하므로 마지막 줄을 통째로 교체합니다.\nCOT_INSTRUCTION = ("이미지에서 근거가 되는 부분을 찾아 간단히 단계적으로 설명한 뒤, "\n                   "마지막 줄에 반드시 `정답: x` 형식으로 a, b, c, d 중 한 글자만 쓰세요.")\n\n\n# E6: 정답은 사진 속 사실이 아니라 여러 교육생이 일치해서 고른 답입니다(응답이 갈린 문항은 제외됨).\n# E1 오답 중 \'라벨 오류\'로 본 문항도 사람 기준으로는 일반인 상식(그리스 문자 -> 라틴어)이나\n# 눈에 띄는 쪽(두 수건 중 앞의 흰 수건)을 고른 것이었습니다. 그 기준을 지시문으로 알려 봅니다.\nCONSENSUS_INSTRUCTION = ("이 문제의 정답은 여러 사람이 같은 사진을 보고 일치해서 고른 답입니다. "\n                         "사진에 적힌 표기를 그대로 따르고, 전문 지식보다 일반적인 상식과 "\n                         "눈에 잘 띄는 정보를 기준으로 사람들이 가장 많이 고를 선지의 "\n                         "영문 소문자 한 글자만 답하세요.")\n\n\ndef make_prompt(row, cot=False, consensus=False):\n    """질문/선지 렌더링. 본문은 모든 실험에서 동일하고 마지막 지시문만 달라집니다."""\n    assert not (cot and consensus), "CoT와 합의 지시문은 함께 쓰지 않습니다"\n    instruction = COT_INSTRUCTION if cot else CONSENSUS_INSTRUCTION if consensus else ANSWER_INSTRUCTION\n    return (f"질문: {row[\'question\']}\\n"\n            f"a. {row[\'a\']}\\nb. {row[\'b\']}\\nc. {row[\'c\']}\\nd. {row[\'d\']}\\n"\n            + instruction)\n\n\ndef parse_choice(raw):\n    """출력이 정확히 선지 한 글자일 때만 인정하고, 나머지는 INVALID로 남깁니다."""\n    raw = re.sub(r"<think>.*?</think>", "", str(raw), flags=re.S).strip()\n    raw = re.sub(r"^(?:정답|답|answer)\\s*[:：]\\s*", "", raw, flags=re.I).strip()\n    m = re.fullmatch(r"(?:\\(?([abcd])\\)?[.。]?)", raw, flags=re.I)\n    return m.group(1).lower() if m else "INVALID"\n\n\n# CoT 모드에서는 system 프롬프트의 \'설명 금지\'를 풀어야 합니다.\nSYSTEM_COT = ("You are a visual question answering assistant. "\n              "Think briefly, then end your reply with a final line "\n              "in the form `정답: x` where x is one of a, b, c, d.")\n\n_FINAL_RE = re.compile(r"(?:정답|답|answer|final answer)\\s*[:：]\\s*\\(?\\s*([abcd])\\b",\n                       re.IGNORECASE)\n_BARE_RE = re.compile(r"(?<![a-z])\\(?([abcd])\\)?(?![a-z])", re.IGNORECASE)\n\n\ndef parse_choice_cot(raw):\n    """CoT 출력에서 최종 선지를 추출합니다.\n\n    parse_choice()는 출력이 정확히 한 글자일 때만 인정하므로 CoT 문장은 전부 INVALID가 됩니다.\n    여기서는 `정답: x` 형태를 우선 찾고, 없으면 마지막 줄의 단독 a~d를 씁니다.\n    추론 과정에 등장하는 선지 언급을 최종 답으로 오인하지 않도록 뒤에서부터 찾습니다.\n    """\n    text = re.sub(r"<think>.*?</think>", " ", str(raw), flags=re.S)\n    matches = _FINAL_RE.findall(text)\n    if matches:\n        return matches[-1].lower()\n    # `정답:` 표기가 없으면 마지막 비어 있지 않은 줄에서만 단독 글자를 찾습니다.\n    for line in reversed([l.strip() for l in text.splitlines() if l.strip()]):\n        found = _BARE_RE.findall(line)\n        if len(found) == 1:\n            return found[0].lower()\n        if found:\n            break            # 한 줄에 여러 선지가 있으면 최종 답으로 볼 수 없습니다.\n    return "INVALID"\n\n\n# --- E3: 선지를 감추고 먼저 읽게 하는 2단계 프롬프트 -------------------------\n# E1/E2는 질문과 선지 4개를 한꺼번에 줍니다. 그러면 선지가 판독을 유도합니다.\n# 실제로 E1 오답 중 CoT로도 못 고친 22건에서 정답 문자열이 추론문에 등장한 것은 5건뿐이고,\n# 모델은 간판의 \'미술랭투스타\'를 흔한 표기 \'미슐랭 스타\'로 교정해 읽었습니다.\n# 1단계에서 선지를 감추면 그 닻이 사라지는지 보는 것이 E3의 가설입니다.\n\nSYSTEM_READ = ("You are a careful transcription assistant. "\n               "Write down exactly what is printed in the image. "\n               "Never correct spelling, never normalize to a more common wording, never guess.")\n\nREAD_INSTRUCTION = ("위 질문에 답하는 데 필요한 부분의 글자를 이미지에 적힌 그대로 옮겨 적으세요. "\n                    "맞춤법을 고치거나 더 흔한 표현으로 바꾸지 마세요. "\n                    "글자가 보이지 않으면 \'확인 불가\'라고만 쓰세요.")\n\nMATCH_INSTRUCTION = "전사 내용과 이미지를 함께 보고 정답 선지의 영문 소문자 한 글자만 답하세요."\n\n\ndef make_read_prompt(row):\n    """1단계. 선지를 넣지 않습니다. 넣는 순간 실험이 무의미해집니다."""\n    return f"질문: {row[\'question\']}\\n" + READ_INSTRUCTION\n\n\ndef make_match_prompt(row, transcript):\n    """2단계. 1단계 전사를 앞에 붙이고 나머지는 E1과 동일하게 렌더링합니다."""\n    return (f"이미지에서 읽은 글자: {str(transcript).strip()}\\n\\n"\n            f"질문: {row[\'question\']}\\n"\n            f"a. {row[\'a\']}\\nb. {row[\'b\']}\\nc. {row[\'c\']}\\nd. {row[\'d\']}\\n"\n            + MATCH_INSTRUCTION)\n\n\nIMAGE_MAX_EDGE = 960  # 원본 노트북의 공통 이미지 규칙. B와 C가 반드시 같은 값을 써야 비교가 성립합니다.\n\n\ndef open_image(row, max_edge=IMAGE_MAX_EDGE):\n    """원본 노트북 open_image와 동일: RGB 변환 후 긴 변을 max_edge로 제한."""\n    from PIL import Image\n    with Image.open(DATA_DIR / str(row["path"])) as im:\n        im = im.convert("RGB")\n        if max_edge is not None and max(im.size) > max_edge:\n            scale = max_edge / max(im.size)\n            im = im.resize((max(1, round(im.width * scale)), max(1, round(im.height * scale))),\n                           Image.Resampling.LANCZOS)\n        return im\n\n\ndef ids_sha256(ids):\n    return hashlib.sha256(json.dumps(list(ids)).encode()).hexdigest()\n\n\ndef load_val(run_dir=BASELINE_RUN):\n    """기준 실험이 저장한 검증 분할을 그대로 읽고, ID 해시로 동일 분할임을 확인합니다.\n\n    sklearn 버전이 달라져도 재계산하지 않으므로 분할이 어긋날 수 없습니다.\n    """\n    run_dir = Path(run_dir)\n    val_path = run_dir / "train_half_val.csv"\n    if not val_path.is_file():\n        raise FileNotFoundError(\n            f"기준 분할이 없습니다: {val_path}. 먼저 실험 A 노트북의 분할 셀을 실행하세요.")\n    val_df = pd.read_csv(val_path, keep_default_na=False)\n    config_path = run_dir / "run_config.json"\n    if config_path.is_file():\n        expected = json.loads(config_path.read_text(encoding="utf-8")).get("validation_ids_sha256")\n        actual = ids_sha256(val_df.id.tolist())\n        if expected and expected != actual:\n            raise ValueError(f"검증 분할이 기준과 다릅니다: {actual} != {expected}")\n        print(f"검증 분할 해시 일치: {actual[:16]}…")\n    missing = [p for p in val_df.path if not (DATA_DIR / p).is_file()]\n    if missing:\n        raise FileNotFoundError(f"이미지 누락 {len(missing)}개. 예: {missing[:3]}")\n    print(f"검증 문항 {len(val_df)}개 / 정답 분포:\\n{val_df.answer.value_counts().sort_index()}")\n    return val_df\n\n\ndef load_ocr_evidence(run_dir=BASELINE_RUN, mode="ocr_llm"):\n    """실험 A가 남긴 stage1.jsonl(Spotting 결과)을 id -> 텍스트로 읽습니다. 실험 C가 사용."""\n    path = Path(run_dir) / mode / "stage1.jsonl"\n    if not path.is_file():\n        raise FileNotFoundError(f"OCR 결과가 없습니다: {path}. 실험 A를 먼저 완료하세요.")\n    evidence = {}\n    with path.open(encoding="utf-8") as f:\n        for line in f:\n            item = json.loads(line)\n            evidence[item["id"]] = item["stage1_raw"]\n    empty = sum(1 for v in evidence.values() if not v.strip())\n    print(f"OCR 근거 {len(evidence)}건 로드 (빈 결과 {empty}건)")\n    return evidence\n\n\ndef save_analysis(pred_df, name, out_dir, extra=None):\n    """정확도/macro-F1/혼동행렬/분류리포트를 저장하고 지표 dict를 반환합니다."""\n    from sklearn.metrics import (accuracy_score, classification_report,\n                                 confusion_matrix, f1_score)\n    import matplotlib.pyplot as plt\n\n    dest = Path(out_dir)\n    dest.mkdir(parents=True, exist_ok=True)\n    pred_df.to_csv(dest / "validation_predictions.csv", index=False, encoding="utf-8-sig")\n    pred_df.loc[~pred_df["correct"]].to_csv(dest / "validation_errors.csv",\n                                            index=False, encoding="utf-8-sig")\n    # INVALID를 열로 유지하여 오답이 사라지거나 임의로 a가 되지 않도록 합니다.\n    labels = CHOICES + ["INVALID"]\n    cm = confusion_matrix(pred_df["answer"], pred_df["pred"], labels=labels)\n    cm_df = pd.DataFrame(cm, index=[f"true_{k}" for k in labels],\n                         columns=[f"pred_{k}" for k in labels])\n    cm_df.to_csv(dest / "confusion_matrix.csv", encoding="utf-8-sig")\n    class_df = pd.DataFrame(classification_report(\n        pred_df["answer"], pred_df["pred"], labels=CHOICES,\n        output_dict=True, zero_division=0)).T\n    class_df.to_csv(dest / "classification_report.csv", encoding="utf-8-sig")\n    score = {\n        "experiment": name,\n        "validation_rows": len(pred_df),\n        "correct": int(pred_df.correct.sum()),\n        "accuracy": float(accuracy_score(pred_df.answer, pred_df.pred)),\n        "macro_f1": float(f1_score(pred_df.answer, pred_df.pred, labels=CHOICES,\n                                   average="macro", zero_division=0)),\n        "invalid": int((pred_df.pred == "INVALID").sum()),\n        "invalid_rate": float((pred_df.pred == "INVALID").mean()),\n        "mean_inference_seconds": float(pred_df.seconds.mean()),\n        "total_inference_seconds": float(pred_df.seconds.sum()),\n        "validation_ids_sha256": ids_sha256(pred_df.id.tolist()),\n    }\n    score.update(extra or {})\n    (dest / "metrics.json").write_text(json.dumps(score, ensure_ascii=False, indent=2),\n                                       encoding="utf-8")\n    print(f"[{name}] Accuracy={score[\'accuracy\']:.4f} / macro-F1={score[\'macro_f1\']:.4f} "\n          f"/ INVALID={score[\'invalid\']}")\n    print(class_df.loc[CHOICES, ["precision", "recall", "f1-score", "support"]].round(3))\n    fig, ax = plt.subplots(figsize=(7, 5))\n    ax.imshow(cm[:4, :], cmap="Blues")\n    ax.set_xticks(range(5), labels)\n    ax.set_yticks(range(4), CHOICES)\n    ax.set_xlabel("Predicted")\n    ax.set_ylabel("True")\n    ax.set_title(f"{name} | Accuracy={score[\'accuracy\']:.3f}")\n    for i in range(4):\n        for j in range(5):\n            ax.text(j, i, str(cm[i, j]), ha="center", va="center")\n    fig.tight_layout()\n    fig.savefig(dest / "confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    return score\n\n\ndef _self_test():\n    assert parse_choice("c") == "c"\n    assert parse_choice("a 또는 b") == "INVALID"\n    assert parse_choice("") == "INVALID"\n    assert parse_choice("정답: D") == "d"\n    assert parse_choice("간판을 보면 ... 정답: c") == "INVALID"   # 엄격 파서는 문장을 거부\n    # CoT 파서\n    assert parse_choice_cot("a 와 b 를 비교하면...\\n정답: c") == "c"\n    assert parse_choice_cot("생각: b 인 것 같지만 아니다.\\n정답: (D)") == "d"\n    assert parse_choice_cot("설명이 길다\\n\\nb") == "b"\n    assert parse_choice_cot("a 또는 b 중 하나") == "INVALID"\n    assert parse_choice_cot("") == "INVALID"\n    assert parse_choice_cot("정답: a 이지만 정답: b 로 정정") == "b"   # 마지막 선언을 채택\n    row = {"question": "Q", "a": "A", "b": "B", "c": "C", "d": "D"}\n    assert make_prompt(row).endswith(ANSWER_INSTRUCTION)\n    assert make_prompt(row, cot=True).endswith(COT_INSTRUCTION)\n    # 질문/선지 본문은 두 모드가 완전히 동일해야 비교가 성립합니다.\n    assert make_prompt(row).rsplit(chr(10), 1)[0] == make_prompt(row, cot=True).rsplit(chr(10), 1)[0]\n    assert make_prompt(row, consensus=True).endswith(CONSENSUS_INSTRUCTION)\n    assert make_prompt(row).rsplit(chr(10), 1)[0] == make_prompt(row, consensus=True).rsplit(chr(10), 1)[0]\n    # E3 1단계는 선지가 새면 안 됩니다.\n    read = make_read_prompt(row)\n    assert read.endswith(READ_INSTRUCTION)\n    for key in CHOICES:\n        assert f"{key}. {row[key]}" not in read, "1단계 프롬프트에 선지가 들어갔습니다"\n    # E3 2단계는 전사문만 앞에 붙고 질문/선지 본문은 E1과 동일해야 합니다.\n    match = make_match_prompt(row, "테스트 전사")\n    assert "테스트 전사" in match and match.endswith(MATCH_INSTRUCTION)\n    body = lambda s: s.split("질문: ", 1)[1].rsplit(chr(10), 1)[0]\n    assert body(match) == body(make_prompt(row)), "2단계 질문/선지 본문이 E1과 다릅니다"\n    print("공통 프롬프트/파서 OK (엄격 + CoT + E3 2단계)")\n\n\nif __name__ == "__main__":\n    _self_test()\n'
import hashlib
assert hashlib.sha256(COMMON_EVAL_SRC.encode("utf-8")).hexdigest() == "ff6dc88f6916fa149a3a8fdec7dfa60fa384701c542aa19fedc582ce6f8850ae"
SRC_DIR = Path("/content/src"); SRC_DIR.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "common_eval.py").write_text(COMMON_EVAL_SRC, encoding="utf-8")
sys.path.insert(0, str(SRC_DIR))
import common_eval as ce

ce.DATA_DIR = DATA_ROOT
print("데이터:", DATA_ROOT, "/ 이미지 긴 변 제한:", ce.IMAGE_MAX_EDGE, "px")
print("합의 지시문:", ce.CONSENSUS_INSTRUCTION)

## 3. 모델 적재 (vLLM)

In [ ]:
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3.5-27B"
MAX_NEW_TOKENS = 24          # E1~E4와 동일
LOGPROBS_K = 20

t0 = time.time()
llm = LLM(model=MODEL_ID, dtype="bfloat16", max_model_len=4096, gpu_memory_utilization=0.90,
          limit_mm_per_prompt={"image": 1}, seed=42, download_dir="/content/models")
proc = AutoProcessor.from_pretrained(MODEL_ID)
SAMPLING = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, logprobs=LOGPROBS_K, seed=42)
print(f"적재 {time.time() - t0:.0f}초")


def letter_token_ids():
    tok = proc.tokenizer
    return {ch: sorted({tok.encode(f, add_special_tokens=False)[0]
                        for f in (ch, " " + ch, ch.upper(), " " + ch.upper())
                        if tok.encode(f, add_special_tokens=False)})
            for ch in ce.CHOICES}


LETTERS = letter_token_ids()
print("a~d 토큰 id:", LETTERS)

## 4. 추론 함수

E4 제출과 같은 메시지·이미지 규칙·파서입니다. `consensus=True`면 마지막 지시문만 합의 지시문으로 바뀝니다.
256문항마다 `predictions.jsonl`에 쓰고 Drive에 복사하며, 다시 실행하면 기록된 문항을 건너뜁니다.
첫 토큰의 a~d 로그확률(상위 20개 밖이면 `-1e9`)도 남겨 확신도와 사람 투표 비율을 비교합니다.

In [ ]:
CHUNK = 256
OUT_ROOT = Path("/content/out")


def make_request(row, consensus):
    image = ce.open_image(row)
    messages = [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},
                {"role": "user", "content": [{"type": "image"},
                                             {"type": "text",
                                              "text": ce.make_prompt(row, consensus=consensus)}]}]
    prompt = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                      enable_thinking=False)
    return {"prompt": prompt, "multi_modal_data": {"image": image}}, image.size


def letter_logprobs(first_step):
    out = {}
    for ch, ids in LETTERS.items():
        vals = [first_step[i].logprob for i in ids if i in first_step]
        out[ch] = max(vals) if vals else -1e9
    return out


def run(rows, name, consensus=False):
    out_dir, drive_dir = OUT_ROOT / name, DRIVE_BASE / "output" / name
    out_dir.mkdir(parents=True, exist_ok=True); drive_dir.mkdir(parents=True, exist_ok=True)
    pred_path = out_dir / "predictions.jsonl"
    if not pred_path.is_file() and (drive_dir / "predictions.jsonl").is_file():
        shutil.copy2(drive_dir / "predictions.jsonl", pred_path)
        print("Drive에서 이전 기록을 복구했습니다")
    done = set()
    if pred_path.is_file():
        done = {json.loads(l)["id"] for l in pred_path.open(encoding="utf-8") if l.strip()}
    todo = [r for r in rows if r["id"] not in done]
    print(f"[{name}] 대상 {len(rows)} / 기록됨 {len(done)} / 남은 {len(todo)}")
    t_all = time.time()
    for start in range(0, len(todo), CHUNK):
        batch = todo[start:start + CHUNK]
        t0 = time.time()
        reqs, sizes = zip(*(make_request(r, consensus) for r in batch))
        if start == 0:
            assert "<|image_pad|>" in reqs[0]["prompt"], "프롬프트에 이미지 토큰이 없습니다"
            assert reqs[0]["prompt"].rstrip().count(
                ce.CONSENSUS_INSTRUCTION if consensus else ce.ANSWER_INSTRUCTION) == 1
        outs = llm.generate(list(reqs), SAMPLING, use_tqdm=False)
        elapsed = time.time() - t0
        with pred_path.open("a", encoding="utf-8") as log:
            for row, size, o in zip(batch, sizes, outs):
                c = o.outputs[0]
                raw = c.text.strip()
                log.write(json.dumps(dict(
                    id=row["id"], raw_output=raw, pred=ce.parse_choice(raw),
                    letter_logprobs=letter_logprobs(c.logprobs[0]) if c.logprobs else {},
                    answer_truncated=c.finish_reason == "length", answer_tokens=len(c.token_ids),
                    seconds=elapsed / len(batch), image_width=size[0], image_height=size[1]),
                    ensure_ascii=False) + "\n")
        for f in out_dir.glob("*"):
            if f.is_file():
                shutil.copy2(f, drive_dir / f.name)
        n_done = start + len(batch)
        print(f"  {n_done}/{len(todo)}  {elapsed / len(batch):.3f}초/문항  "
              f"남은 약 {(time.time() - t_all) / n_done * (len(todo) - n_done) / 60:.0f}분")
    res = pd.read_json(pred_path, lines=True).drop_duplicates("id", keep="last")
    res["conf"] = res.letter_logprobs.map(lambda lp: math.exp(max(lp.values())) if lp else float("nan"))
    return res


def mcnemar_exact(b, c):
    n = b + c
    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)

## 5. 평가 분할

`splits/e6_split.csv`(SHA-256 `f690bf3f948e96ef…`)의 holdout 1,300과 그 안의 val 672입니다.

In [ ]:
HOLDOUT_NUMS = [37, 57, 61, 65, 72, 77, 78, 84, 91, 93, 106, 108, 110, 111, 113, 117, 119, 124, 127, 130, 132, 134, 143, 145, 155, 156, 166, 169, 174, 177, 185, 192, 193, 203, 205, 208, 209, 213, 214, 221, 224, 225, 226, 230, 231, 234, 238, 239, 241, 245, 247, 251, 252, 260, 266, 267, 271, 273, 277, 280, 281, 290, 299, 300, 303, 304, 307, 309, 312, 316, 317, 319, 322, 323, 325, 335, 339, 346, 347, 355, 356, 360, 363, 365, 370, 371, 381, 387, 388, 390, 401, 428, 432, 442, 463, 475, 480, 481, 484, 500, 505, 508, 515, 517, 519, 524, 528, 529, 531, 537, 542, 551, 558, 567, 569, 570, 573, 574, 575, 584, 594, 599, 601, 603, 613, 614, 625, 641, 649, 650, 652, 670, 673, 675, 698, 703, 718, 723, 742, 746, 756, 758, 765, 775, 776, 778, 785, 786, 787, 792, 797, 803, 807, 827, 829, 833, 842, 846, 849, 858, 863, 870, 871, 873, 875, 882, 893, 909, 912, 913, 937, 939, 940, 944, 949, 968, 979, 983, 985, 1000, 1003, 1005, 1006, 1012, 1018, 1022, 1033, 1034, 1039, 1042, 1045, 1048, 1049, 1052, 1061, 1065, 1070, 1072, 1075, 1079, 1080, 1083, 1087, 1092, 1094, 1097, 1106, 1116, 1117, 1118, 1119, 1120, 1123, 1127, 1134, 1139, 1145, 1149, 1150, 1152, 1155, 1156, 1166, 1171, 1172, 1174, 1180, 1181, 1184, 1193, 1197, 1199, 1200, 1201, 1211, 1213, 1218, 1221, 1223, 1224, 1230, 1240, 1272, 1274, 1275, 1281, 1282, 1287, 1289, 1297, 1298, 1315, 1324, 1325, 1328, 1329, 1333, 1339, 1341, 1342, 1348, 1352, 1361, 1368, 1372, 1373, 1383, 1390, 1391, 1392, 1402, 1404, 1405, 1408, 1420, 1421, 1423, 1429, 1433, 1434, 1437, 1454, 1457, 1458, 1460, 1461, 1471, 1475, 1483, 1500, 1512, 1520, 1525, 1528, 1530, 1531, 1533, 1535, 1537, 1539, 1540, 1543, 1545, 1555, 1560, 1563, 1566, 1578, 1580, 1583, 1593, 1599, 1604, 1608, 1609, 1614, 1615, 1621, 1626, 1641, 1649, 1650, 1651, 1659, 1663, 1670, 1682, 1700, 1705, 1713, 1725, 1729, 1730, 1733, 1734, 1737, 1744, 1752, 1753, 1754, 1755, 1758, 1762, 1770, 1771, 1772, 1781, 1802, 1803, 1811, 1824, 1825, 1826, 1827, 1828, 1837, 1860, 1861, 1863, 1867, 1871, 1874, 1884, 1888, 1891, 1892, 1893, 1894, 1896, 1903, 1906, 1910, 1913, 1914, 1919, 1923, 1929, 1930, 1931, 1933, 1946, 1954, 1960, 1968, 1973, 1974, 1975, 1977, 1981, 1982, 1987, 1988, 1991, 1992, 1995, 1996, 1997, 2005, 2009, 2012, 2014, 2018, 2031, 2041, 2046, 2049, 2060, 2065, 2078, 2079, 2085, 2087, 2091, 2094, 2102, 2105, 2107, 2120, 2131, 2132, 2133, 2144, 2149, 2150, 2152, 2161, 2162, 2166, 2167, 2172, 2173, 2177, 2180, 2189, 2192, 2198, 2206, 2210, 2214, 2216, 2222, 2229, 2233, 2241, 2263, 2268, 2270, 2273, 2286, 2288, 2300, 2308, 2313, 2315, 2324, 2328, 2344, 2346, 2349, 2352, 2353, 2356, 2363, 2364, 2366, 2368, 2384, 2385, 2390, 2415, 2430, 2436, 2441, 2445, 2449, 2450, 2451, 2455, 2458, 2461, 2467, 2468, 2470, 2475, 2481, 2482, 2484, 2485, 2487, 2491, 2498, 2510, 2514, 2515, 2517, 2519, 2522, 2527, 2536, 2538, 2546, 2548, 2559, 2561, 2564, 2565, 2581, 2583, 2586, 2587, 2590, 2610, 2623, 2625, 2628, 2629, 2652, 2658, 2659, 2662, 2668, 2669, 2671, 2691, 2697, 2700, 2704, 2706, 2710, 2721, 2725, 2730, 2731, 2736, 2741, 2743, 2749, 2757, 2771, 2772, 2777, 2782, 2784, 2786, 2797, 2798, 2810, 2812, 2814, 2815, 2816, 2817, 2830, 2836, 2842, 2852, 2864, 2865, 2866, 2877, 2886, 2897, 2901, 2907, 2909, 2916, 2925, 2928, 2931, 2936, 2937, 2943, 2967, 2972, 2976, 2977, 2993, 3003, 3007, 3011, 3013, 3020, 3045, 3049, 3054, 3071, 3073, 3076, 3082, 3083, 3086, 3100, 3113, 3115, 3120, 3121, 3128, 3147, 3154, 3170, 3189, 3190, 3194, 3199, 3202, 3208, 3213, 3221, 3228, 3230, 3239, 3242, 3244, 3250, 3266, 3275, 3280, 3281, 3289, 3295, 3298, 3305, 3311, 3314, 3323, 3328, 3333, 3338, 3349, 3359, 3361, 3363, 3364, 3368, 3374, 3383, 3397, 3398, 3408, 3411, 3415, 3433, 3441, 3447, 3448, 3451, 3452, 3455, 3458, 3459, 3470, 3489, 3490, 3512, 3518, 3540, 3541, 3547, 3548, 3549, 3556, 3560, 3562, 3569, 3580, 3582, 3583, 3589, 3594, 3599, 3616, 3622, 3623, 3628, 3632, 3658, 3663, 3665, 3673, 3676, 3680, 3682, 3688, 3692, 3696, 3698, 3699, 3718, 3724, 3732, 3738, 3742, 3744, 3746, 3757, 3767, 3768, 3781, 3792, 3794, 3796, 3802, 3807, 3809, 3821, 3832, 3847, 3848, 3855, 3867, 3875, 3888, 3890, 3899, 3910, 3911, 3912, 3916, 3920, 3930, 3937, 3939, 3943, 3947, 3952, 3953, 3973, 3977, 3979, 3981, 3985, 3992, 3996, 4001, 4002, 4004, 4015, 4018, 4030, 4033, 4035, 4039, 4044, 4050, 4059, 4063, 4064, 4066, 4068, 4071, 4074, 4082, 4106, 4108, 4116, 4117, 4122, 4125, 4128, 4130, 4132, 4141, 4149, 4150, 4163, 4165, 4167, 4186, 4190, 4191, 4192, 4193, 4194, 4195, 4196, 4202, 4204, 4208, 4210, 4213, 4214, 4222, 4223, 4225, 4229, 4230, 4237, 4241, 4246, 4254, 4257, 4258, 4264, 4267, 4274, 4287, 4288, 4291, 4293, 4297, 4304, 4305, 4306, 4310, 4315, 4320, 4332, 4338, 4339, 4341, 4354, 4360, 4366, 4367, 4372, 4376, 4379, 4380, 4384, 4391, 4392, 4395, 4398, 4401, 4402, 4404, 4405, 4408, 4409, 4415, 4419, 4430, 4439, 4445, 4449, 4450, 4451, 4452, 4454, 4456, 4461, 4462, 4468, 4471, 4478, 4481, 4485, 4488, 4491, 4497, 4509, 4510, 4518, 4522, 4528, 4531, 4539, 4540, 4553, 4555, 4569, 4574, 4580, 4584, 4595, 4596, 4600, 4602, 4603, 4605, 4609, 4614, 4625, 4630, 4635, 4638, 4639, 4640, 4647, 4648, 4649, 4657, 4659, 4665, 4668, 4670, 4673, 4675, 4682, 4683, 4701, 4705, 4708, 4717, 4719, 4720, 4721, 4722, 4723, 4729, 4736, 4738, 4742, 4744, 4748, 4749, 4753, 4754, 4755, 4759, 4760, 4764, 4771, 4780, 4786, 4789, 4790, 4792, 4793, 4794, 4800, 4801, 4802, 4806, 4808, 4809, 4812, 4814, 4815, 4822, 4824, 4826, 4827, 4831, 4836, 4840, 4843, 4847, 4848, 4849, 4854, 4855, 4859, 4864, 4866, 4868, 4874, 4878, 4883, 4889, 4907, 4913, 4914, 4917, 4918, 4925, 4926, 4944, 4948, 4953, 4954, 4957, 4968, 4971, 4985, 5009, 5013, 5015, 5031, 5036, 5045, 5049, 5058, 5077, 5080, 5082, 5086, 5090, 5104, 5108, 5109, 5116, 5121, 5126, 5130, 5145, 5148, 5151, 5155, 5158, 5164, 5167, 5169, 5170, 5171, 5179, 5180, 5181, 5182, 5185, 5187, 5188, 5200, 5209, 5217, 5220, 5223, 5224, 5245, 5250, 5254, 5261, 5266, 5268, 5269, 5270, 5271, 5279, 5281, 5285, 5289, 5290, 5308, 5315, 5316, 5317, 5319, 5331, 5340, 5344, 5361, 5371, 5374, 5378, 5381, 5383, 5384, 5388, 5394, 5397, 5399, 5403, 5420, 5422, 5429, 5441, 5445, 5447, 5452, 5456, 5459, 5461, 5464, 5467, 5478, 5483, 5486, 5488, 5495, 5499, 5505, 5506, 5509, 5517, 5518, 5519, 5521, 5524, 5525, 5526, 5532, 5534, 5536, 5540, 5541, 5542, 5545, 5547, 5555, 5556, 5557, 5560, 5564, 5572, 5577, 5593, 5596, 5601, 5605, 5610, 5618, 5622, 5629, 5637, 5639, 5653, 5657, 5682, 5683, 5684, 5693, 5715, 5723, 5729, 5730, 5733, 5747, 5758, 5775, 5776, 5789, 5790, 5798, 5809, 5811, 5818, 5820, 5821, 5828, 5832, 5833, 5844, 5845, 5855, 5857, 5861, 5862, 5867, 5874, 5875, 5880, 5885, 5888, 5890, 5895, 5904, 5906, 5909, 5911, 5915, 5927, 5928, 5931, 5936, 5941, 5942, 5943, 5950, 5956, 5962, 5970, 5972, 5981, 5982, 5983, 5992, 6001, 6008, 6017, 6021, 6025, 6031, 6034, 6037, 6040, 6050, 6052, 6059, 6065, 6067, 6069, 6072, 6080, 6084, 6100, 6101, 6113, 6114, 6129, 6131, 6138, 6143, 6146, 6149, 6151, 6154, 6157, 6160, 6162, 6168, 6169, 6171, 6179, 6182, 6189, 6196, 6200, 6204, 6211, 6212, 6216, 6217, 6223, 6241, 6250, 6259, 6260, 6265, 6267, 6269, 6283, 6284, 6287, 6288, 6290, 6302, 6308, 6315, 6323, 6340, 6347, 6352, 6359, 6371, 6374, 6375, 6377, 6378, 6390, 6398, 6404, 6407, 6413, 6416, 6418, 6422, 6426, 6427, 6438, 6445, 6461, 6464, 6476, 6477, 6487, 6488, 6489, 6490, 6495, 6503, 6509, 6522, 6528, 6531, 6537, 6544, 6547, 6548, 6549, 6551, 6556, 6567, 6570, 6571, 6576, 6581, 6582, 6584, 6586, 6588, 6590, 6594, 6603, 6607, 6611, 6613, 6618, 6631, 6633, 6635, 6637, 6643, 6645, 6647, 6652, 6663, 6664, 6667, 6671, 6672, 6673, 6674, 6675, 6679, 6684, 6690, 6693, 6702, 6703, 6707]
VAL672_NUMS = [37, 57, 65, 84, 91, 108, 113, 119, 124, 130, 132, 134, 145, 169, 174, 177, 193, 213, 221, 224, 225, 247, 251, 267, 273, 281, 300, 307, 309, 316, 317, 322, 356, 360, 363, 370, 388, 390, 401, 428, 432, 442, 481, 500, 508, 517, 528, 537, 551, 558, 567, 584, 594, 601, 603, 613, 641, 649, 650, 670, 673, 675, 698, 703, 756, 775, 778, 785, 787, 792, 797, 803, 807, 827, 833, 842, 846, 858, 863, 870, 871, 873, 893, 909, 912, 913, 937, 944, 968, 983, 985, 1000, 1005, 1006, 1012, 1034, 1039, 1042, 1045, 1049, 1061, 1072, 1079, 1080, 1092, 1094, 1097, 1106, 1117, 1118, 1120, 1127, 1139, 1145, 1149, 1152, 1155, 1156, 1166, 1171, 1172, 1193, 1199, 1200, 1201, 1213, 1272, 1275, 1297, 1298, 1324, 1325, 1328, 1329, 1333, 1341, 1342, 1348, 1361, 1372, 1392, 1402, 1420, 1433, 1437, 1458, 1461, 1471, 1483, 1512, 1525, 1528, 1530, 1533, 1537, 1543, 1560, 1578, 1593, 1614, 1641, 1650, 1659, 1663, 1705, 1725, 1730, 1734, 1744, 1753, 1755, 1758, 1762, 1770, 1803, 1825, 1827, 1828, 1837, 1861, 1867, 1888, 1891, 1892, 1893, 1894, 1903, 1919, 1930, 1931, 1954, 1960, 1977, 1981, 1982, 1992, 1997, 2014, 2018, 2031, 2046, 2049, 2060, 2079, 2085, 2087, 2105, 2120, 2133, 2144, 2149, 2172, 2173, 2177, 2189, 2198, 2206, 2216, 2222, 2241, 2263, 2288, 2308, 2313, 2315, 2324, 2344, 2346, 2349, 2352, 2353, 2356, 2363, 2364, 2366, 2368, 2384, 2415, 2436, 2449, 2458, 2467, 2470, 2481, 2487, 2498, 2514, 2517, 2527, 2538, 2546, 2548, 2561, 2565, 2583, 2586, 2590, 2628, 2662, 2668, 2669, 2697, 2704, 2725, 2730, 2731, 2736, 2741, 2743, 2749, 2771, 2777, 2784, 2797, 2812, 2816, 2817, 2830, 2836, 2852, 2864, 2865, 2907, 2925, 2928, 2931, 2937, 2967, 2972, 2993, 3003, 3011, 3013, 3045, 3054, 3073, 3076, 3083, 3086, 3100, 3113, 3115, 3121, 3128, 3154, 3194, 3199, 3202, 3228, 3230, 3239, 3244, 3250, 3266, 3280, 3289, 3298, 3323, 3328, 3333, 3349, 3359, 3361, 3364, 3374, 3383, 3408, 3441, 3447, 3448, 3451, 3452, 3455, 3458, 3470, 3489, 3541, 3549, 3580, 3582, 3583, 3594, 3599, 3616, 3622, 3623, 3632, 3658, 3663, 3665, 3673, 3676, 3680, 3688, 3696, 3698, 3699, 3718, 3757, 3792, 3802, 3809, 3821, 3832, 3875, 3888, 3890, 3899, 3912, 3916, 3920, 3930, 3943, 3947, 3973, 3977, 3981, 3985, 3996, 4018, 4039, 4044, 4068, 4074, 4082, 4106, 4117, 4125, 4130, 4132, 4149, 4167, 4186, 4191, 4193, 4195, 4204, 4208, 4213, 4229, 4254, 4257, 4264, 4267, 4291, 4293, 4297, 4305, 4306, 4310, 4315, 4332, 4341, 4354, 4367, 4372, 4380, 4392, 4398, 4401, 4402, 4404, 4408, 4409, 4419, 4430, 4439, 4445, 4450, 4454, 4461, 4468, 4471, 4478, 4491, 4497, 4518, 4528, 4531, 4540, 4553, 4569, 4584, 4596, 4600, 4602, 4603, 4605, 4614, 4630, 4639, 4649, 4657, 4659, 4668, 4670, 4701, 4708, 4719, 4720, 4744, 4748, 4754, 4760, 4764, 4771, 4786, 4789, 4790, 4801, 4802, 4808, 4812, 4814, 4822, 4824, 4831, 4836, 4843, 4847, 4859, 4866, 4868, 4874, 4878, 4889, 4914, 4917, 4918, 4925, 4944, 4954, 4968, 4971, 5009, 5013, 5045, 5058, 5077, 5080, 5109, 5126, 5145, 5148, 5155, 5158, 5167, 5169, 5182, 5187, 5188, 5209, 5220, 5245, 5254, 5261, 5266, 5268, 5269, 5285, 5289, 5290, 5308, 5331, 5340, 5361, 5371, 5374, 5384, 5420, 5422, 5452, 5459, 5461, 5478, 5486, 5488, 5495, 5499, 5506, 5509, 5518, 5519, 5521, 5524, 5525, 5526, 5532, 5534, 5536, 5545, 5547, 5555, 5556, 5560, 5564, 5610, 5618, 5622, 5629, 5653, 5683, 5684, 5723, 5729, 5730, 5747, 5789, 5790, 5811, 5818, 5832, 5833, 5844, 5845, 5855, 5857, 5861, 5867, 5880, 5885, 5888, 5895, 5904, 5906, 5928, 5936, 5950, 5962, 5972, 5982, 5992, 6017, 6025, 6031, 6037, 6040, 6050, 6069, 6100, 6101, 6113, 6131, 6143, 6149, 6154, 6157, 6160, 6162, 6169, 6171, 6179, 6189, 6196, 6212, 6216, 6217, 6241, 6259, 6260, 6267, 6283, 6284, 6287, 6288, 6290, 6302, 6308, 6315, 6323, 6340, 6347, 6359, 6371, 6375, 6378, 6390, 6404, 6413, 6416, 6418, 6422, 6426, 6427, 6438, 6445, 6464, 6476, 6522, 6528, 6547, 6548, 6549, 6551, 6556, 6576, 6584, 6590, 6603, 6611, 6618, 6637, 6645, 6647, 6663, 6671, 6673, 6675, 6684, 6693, 6702, 6703]
HOLDOUT = {f"train_{n:04d}.jpg" for n in HOLDOUT_NUMS}
VAL672_LIST = [f"train_{n:04d}.jpg" for n in VAL672_NUMS]
assert ce.ids_sha256(VAL672_LIST) == "0aad93d6210e1b23f94186edebead4cd754bf0f93609d59f06f67c5489791cfe", "val 672가 기준 분할과 다릅니다"
VAL672 = set(VAL672_LIST)
assert len(HOLDOUT) == 1300 and VAL672 <= HOLDOUT

train_df = pd.read_csv(DATA_ROOT / "train.csv", keep_default_na=False)
train_df["split"] = train_df.id.map(lambda i: "holdout" if i in HOLDOUT else "train")
print(train_df.split.value_counts().to_dict())


def report(res, title):
    t = train_df[["id", "answer", "split"]].merge(res[["id", "pred", "conf"]], on="id",
                                                  validate="one_to_one")
    t["ok"] = t.pred == t.answer
    print(f"=== {title} ===")
    for name, part in [("전체 6,714", t), ("학습 5,414", t[t.split == "train"]),
                       ("holdout 1,300", t[t.split == "holdout"]), ("val 672", t[t.id.isin(VAL672)])]:
        print(f"  {name:14s} {part.ok.mean():.4f} ({int(part.ok.sum())}/{len(part)})")
    low = t.conf <= 0.8
    print(f"  INVALID {int((t.pred == 'INVALID').sum())} / 확신도 0.8 이하 {int(low.sum())}문항"
          f"({low.mean():.1%}, 정확도 {t[low].ok.mean():.3f}) / 그 밖 정확도 {t[~low].ok.mean():.4f}")
    wrong = t[~t.ok]
    print(f"  사람과 어긋난 {len(wrong)}문항 중 확신도 0.95 초과 {int((wrong.conf > 0.95).sum())}"
          " (모델이 확신하는데 사람과 다른 문항 = 학습 데이터 진단 대상)")
    return t

## 6. [1단계] train 6,714 제로샷 — 새 기준선

val 672 값이 로컬 E4(96.88%)와 같거나 아주 가까워야 정상입니다(E4 제출 때 일치율 98.8%).

In [ ]:
base = run(train_df.drop(columns=["answer", "split"]).to_dict("records"), "exp_E6_zeroshot_train")
base_t = report(base, "27B 제로샷 (E4와 같은 지시문)")

## 7. [2단계] dev 2,683 제로샷 — 사람 투표와 비교

dev는 문항당 최대 5명의 응답이 있고 최다 응답이 3표 또는 2표뿐입니다. 진짜 사람 응답이라면
모델은 **3표 문항에서 2표 문항보다 다수 응답과 더 자주 일치**해야 하고, 모델 확신도도
투표가 몰릴수록 높아야 합니다. 이 관계가 보이면 dev를 모호한 사례의 학습 자료(다수결 또는 투표 비율)로
쓸 근거가 됩니다.

In [ ]:
dev_df = pd.read_csv(DATA_ROOT / "dev.csv", keep_default_na=False)
VOTES = [f"answer{i}" for i in range(1, 6)]
dev_res = run(dev_df.drop(columns=VOTES).to_dict("records"), "exp_E6_zeroshot_dev")

d = dev_df.merge(dev_res[["id", "pred", "conf", "letter_logprobs"]], on="id", validate="one_to_one")
d["votes"] = d[VOTES].apply(lambda r: [v for v in r if v in ce.CHOICES], axis=1, result_type="reduce")
d["n"] = d.votes.map(len)
d["top"] = d.votes.map(lambda v: max(set(v), key=v.count) if v else "")
d["top_k"] = d.apply(lambda r: r.votes.count(r.top) if r.n else 0, axis=1)
d["tie"] = d.apply(lambda r: sum(r.votes.count(x) == r.top_k for x in set(r.votes)) > 1 if r.n else True, axis=1)
d["model_share"] = d.apply(lambda r: r.votes.count(r.pred) / r.n if r.n else float("nan"), axis=1)
d["match_top"] = (d.pred == d.top) & ~d.tie

print("=== dev: 최다 응답 표 수별 ===")
g = d[d.n == 5].groupby("top_k").agg(문항=("id", "size"), 다수와일치=("match_top", "mean"),
                                     모델답의득표율=("model_share", "mean"), 평균확신도=("conf", "mean"))
print(g.round(3).to_string())
print(f"\n모델 답이 한 표도 못 받은 문항 {int((d.model_share == 0).sum())} / {len(d)}")
dev_votes_path = OUT_ROOT / "exp_E6_zeroshot_dev" / "dev_with_votes.jsonl"
d.drop(columns=["letter_logprobs"]).to_json(dev_votes_path, orient="records", lines=True, force_ascii=False)
shutil.copy2(dev_votes_path, DRIVE_BASE / "output" / "exp_E6_zeroshot_dev" / dev_votes_path.name)

## 8. [3단계] train 6,714 + 합의 지시문

지시문 하나만 바꿨으므로 1단계와 문항 단위로 맞대 McNemar로 봅니다. 6,714문항이면 0.5%p 안팎의
차이도 가려집니다. 판단은 **전체**가 아니라 **holdout** 기준으로 하세요. 이후 QLoRA와 같은 잣대로
비교해야 하기 때문입니다(학습이 없으니 전체 수치도 참고로는 유효합니다).

In [ ]:
cons = run(train_df.drop(columns=["answer", "split"]).to_dict("records"), "exp_E6_consensus_train",
           consensus=True)
cons_t = report(cons, "27B 제로샷 + 합의 지시문")

j = base_t[["id", "answer", "split", "ok"]].merge(cons_t[["id", "ok"]], on="id", suffixes=("_base", "_cons"))
for name, part in [("전체 6,714", j), ("holdout 1,300", j[j.split == "holdout"])]:
    b = int((part.ok_base & ~part.ok_cons).sum()); c = int((~part.ok_base & part.ok_cons).sum())
    diff = (c - b) / len(part); se = math.sqrt(b + c) / len(part)
    print(f"[{name}] 기존만 정답 {b} / 합의만 정답 {c} / 차이 {diff * 100:+.2f}pp "
          f"95%CI [{(diff - 1.96 * se) * 100:+.2f}, {(diff + 1.96 * se) * 100:+.2f}] / McNemar p = {mcnemar_exact(b, c):.4g}")

## 끝나면

Drive의 `output/exp_E6_zeroshot_train`, `exp_E6_zeroshot_dev`, `exp_E6_consensus_train`을
로컬 `output/`으로 받아 주세요. 다음 단계에서 모델–사람 불일치 문항을 유형별로 나누고
QLoRA 학습 데이터 기준(전부 / 확신 높은 불일치 제외 / dev 추가)을 정합니다.